# 얘도 분류 들어갈 것 같은디...
- 네, 늘 하던 그거요.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import PCA # 주성분분석
from sklearn.preprocessing import StandardScaler # 마 서케일러다 안카요
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report, confusion_matrix, ConfusionMatrixDisplay
from statsmodels.stats.outliers_influence import variance_inflation_factor # VIF
from sklearn.model_selection import train_test_split

from scipy.stats import skew
import optuna
from xgboost import XGBRegressor # 정말 뜬금없이 XGBoost
import shap

In [ ]:
# 그래프 기본 테마 설정
sns.set_theme(palette="viridis", style="whitegrid", font_scale=1)
sns.color_palette("viridis", as_cmap=True)

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Mabinogi_classic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 9
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mirichoi0218/insurance")

print("Path to dataset files:", path)

In [ ]:
cost = pd.read_csv(f'{path}/insurance.csv')

# 정보 확인
## .info()

In [ ]:
cost.info()

## .describe()

In [ ]:
cost.describe()

In [ ]:
cost.describe(include='O')

## .isna().sum()

In [ ]:
cost.isna().sum() # 오 굿

## .columns

In [ ]:
cost.columns

## .head()

In [ ]:
cost.head()

- Charges: 보험료(고객이 내는)

그래서 이 데이터 전처리 순서가 어떻게 되냐면
1. 성별 및 흡연유무: 인코딩
2. 자녀: 유/무로 범주화
3. 지역: 너 나가
4. bmi, age: 보자 스케일러가...

이렇게 됩니다. Charges는 우리가 예측해야 하는 데이터임다.

# 전처리
## 자녀 수 범주화
- 자녀가 있으면 1, 없으면 0

In [ ]:
# 자녀 수는 모르겠고 유무로 범주화할거임. ㅇㅋ? ㅇㅇㅋ.
cost['have_child'] = cost['children'].apply(lambda x: 0 if x == 0 else 1) # 자녀수가 0이면 0, 0보다 크면 1
# 유자녀가 1입니다

In [ ]:
cost

## 지역 나가

In [ ]:
cost.drop('region', axis = 1, inplace = True)

In [ ]:
cost

## 마! 서케일러!

In [ ]:
scaler = StandardScaler()

cost[['age','bmi']] = scaler.fit_transform(cost[['age','bmi']])

In [ ]:
cost

## 인코딩 & 매핑

In [ ]:
# 흡연여부
cost['smoker'] = cost['smoker'].map({'yes': 1, 'no': 0})

In [ ]:
cost

In [ ]:
cost['sex'] = cost['sex'].map({'male': 1, 'female': 0})

In [ ]:
cost

## 자녀 칼럼 나갓

In [ ]:
cost.drop('children', axis = 1, inplace = True)

In [ ]:
cost

# Optuna
- 알아서 최적화 해주셈 할 때 쓰십셔..

In [ ]:
# 1. 독립변수(X)와 종속변수(y) 분리
X = cost.drop('charges', axis=1)
y = np.log1p(cost['charges']) # 아까 말씀드린 로그 변환!

# 2. 데이터 분할 (보통 8:2나 7.5:2.5로 나눕니다)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"학습 데이터 개수: {len(X_train)}")
print(f"테스트 데이터 개수: {len(X_test)}")

In [ ]:
def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42
    }

    model = XGBRegressor(**param)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return rmse

# 최적화 시작
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50) # 50번 정도만 돌려봐도 감이 옵니다.

print(f"최고의 파라미터: {study.best_params}")

In [ ]:
model = XGBRegressor(**study.best_params)
model.fit(X_train, y_train)

In [ ]:
# 1. 예측 수행 (로그 상태)
y_pred_log = model.predict(X_test)

# 2. 역변환 (로그 -> 원래 달러 단위)
y_pred = np.expm1(y_pred_log)
y_actual = np.expm1(y_test)

# 3. 성능 지표 확인
print(f"R² Score (결정계수): {r2_score(y_actual, y_pred):.4f}")
print(f"MAE (평균 절대 오차): ${mean_absolute_error(y_actual, y_pred):.2f}")
print(f"MSE (평균 제곱 오차): ${mean_squared_error(y_actual, y_pred):.2f}")
print(f"RMSE (제곱근 평균 제곱 오차): ${np.sqrt(mean_squared_error(y_actual, y_pred)):.2f}")

- ??????????????? 머여 설명력이 너무 예상밖인데? 

## 그 분포좀 봅시다

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(cost['charges'], kde=True, color='teal')
plt.title('보험료(Charges) 분포')
plt.xlabel('보험료 ($)')
plt.ylabel('빈도')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(np.log1p(cost['charges']), kde=True, color='teal')
plt.title('보험료(Charges) 분포')
plt.xlabel('보험료 ($)')
plt.ylabel('빈도')
plt.show()

In [ ]:
print(f"왜도(Skewness): {cost['charges'].skew():.4f}")
print(f"왜도(Skewness): {np.log1p(cost['charges']).skew():.4f}")

- 와 왜도 에반데